In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from text_preprocessing.tokenization import hf_token


d:\@FYP-IntentClassifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from training import BANKING77_LABELS

In [3]:
 
def mean_pool(embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    return (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

In [4]:
class LinearProbe(nn.Module):
    def __init__(self, hidden_size: int = 768, num_classes: int = 77):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_classes)
 
    def forward(self, embeddings, attention_mask):
        pooled = mean_pool(embeddings, attention_mask)
        return self.classifier(pooled)
 

In [5]:
class LinearProbePredictor:
    def __init__(
        self,
        checkpoint: str = "best_linear_probe.pt",
        max_length: int = 64,
        device: str | None = None,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
 
        print("Loading tokenizer…")
        self.tokenizer = BertTokenizer.from_pretrained(
            "bert-base-uncased", token=hf_token
        )
 
        print("Loading BERT…")
        self.bert = BertModel.from_pretrained(
            "bert-base-uncased", token=hf_token
        ).to(self.device)
        self.bert.eval()
 
        print(f"Loading linear probe from '{checkpoint}'…")
        self.probe = LinearProbe().to(self.device)
        self.probe.load_state_dict(
            torch.load(checkpoint, map_location=self.device, weights_only=True)
        )
        self.probe.eval()
        print(f"Ready on {self.device}\n")
 
    @torch.no_grad()
    def predict(self, text: str, top_k: int = 5) -> dict:
        # Tokenize
        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = encoded["input_ids"].to(self.device)
        attention_mask = encoded["attention_mask"].to(self.device)
 
        # BERT embeddings
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state  # (1, seq_len, 768)
 
        # Linear probe
        logits = self.probe(embeddings, attention_mask)          # (1, 77)
        probs  = F.softmax(logits, dim=-1).squeeze(0)            # (77,)
 
        top_probs, top_idxs = probs.topk(top_k)
 
        return {
            "query":      text,
            "prediction": BANKING77_LABELS[top_idxs[0].item()],
            "confidence": top_probs[0].item(),
            "top_k": [
                {"label": BANKING77_LABELS[i.item()], "confidence": p.item()}
                for p, i in zip(top_probs, top_idxs)
            ],
        }

In [45]:
def display_result(result: dict, top_k: int = 5) -> None:
    top1_label = result["prediction"]

    print(f"\nInput: '{result['query']}'")
    print(f"{'Rank':<6} {'Intent':<40} {'Confidence'}")
    print("-" * 55)
    for i, item in enumerate(result["top_k"], 1):
        is_top1 = item["label"] == top1_label
        marker = "" if is_top1 else "  NO!"
        print(f"  {i:<4} {item['label']:<40} {item['confidence']*100:.2f}%{marker}")
    print(f"\n('{top1_label}', {result['confidence']:.16f})")

In [46]:
def main():
    predictor = LinearProbePredictor(checkpoint="best_linear_probe.pt")
 
    print("Banking Intent Classifier — Linear Probe")
    
    result = predictor.predict('''my card is not blocked but it keeps getting declined''', top_k=5)
    display_result(result)


In [47]:
if __name__ == "__main__":
    main()

Loading tokenizer…
Loading BERT…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 496.79it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading linear probe from 'best_linear_probe.pt'…
Ready on cuda

Banking Intent Classifier — Linear Probe

Input: 'my card is not blocked but it keeps getting declined'
Rank   Intent                                   Confidence
-------------------------------------------------------
  1    declined_card_payment                    43.97%
  2    declined_transfer                        26.60%  NO!
  3    declined_cash_withdrawal                 13.47%  NO!
  4    top_up_by_bank_transfer_charge           4.78%  NO!
  5    pending_cash_withdrawal                  3.25%  NO!

('declined_card_payment', 0.4396980106830597)
